In [11]:
import folium
import geopandas as gpd
from sklearn.cluster import HDBSCAN
from shapely import concave_hull
from shapely.geometry import MultiPoint

In [12]:
gdf = gpd.read_file("../../data/processed/philly_trees.geojson").to_crs(epsg=2272)
gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

In [13]:
MIN_SPECIES_COUNT = 40
MIN_TREES = 15
MIN_DENSITY = 5 / 43560
HULL_RATIO = 0.15
HDB_MIN_CLUSTER_SIZE = 20
HDB_MIN_SAMPLES = 5

POLY_COLOR = "#e6550d"
DOT_IN = "#a63603"
DOT_OUT = "#000000"

cluster_records = []
included_idx = {}

for sp in gdf["scientific_name"].value_counts().index:
    sub = gdf[gdf["scientific_name"] == sp]
    if len(sub) < MIN_SPECIES_COUNT:
        continue

    coords = sub[["x", "y"]].values
    labels = HDBSCAN(
        min_cluster_size=HDB_MIN_CLUSTER_SIZE,
        min_samples=HDB_MIN_SAMPLES,
        cluster_selection_method="eom",
    ).fit_predict(coords)

    sub = sub.assign(cluster=labels)
    pts = gpd.GeoDataFrame(
        sub, geometry=gpd.points_from_xy(sub["x"], sub["y"]), crs=gdf.crs
    )

    sp_included = set()
    for cid, grp in pts[pts["cluster"] != -1].groupby("cluster"):
        if len(grp) < MIN_TREES:
            continue
        mp = MultiPoint(list(zip(grp.geometry.x, grp.geometry.y)))
        hull = concave_hull(mp, ratio=HULL_RATIO)
        if hull.is_empty or hull.area == 0:
            continue
        density = len(grp) / hull.area
        if density < MIN_DENSITY:
            continue
        cluster_records.append({
            "scientific_name": sp,
            "tree_count": len(grp),
            "area_sqft": hull.area,
            "density": density,
            "geometry": hull,
        })
        sp_included.update(grp.index)
    if sp_included:
        included_idx[sp] = sp_included

clusters = gpd.GeoDataFrame(cluster_records, crs=gdf.crs)
print(f"HDBSCAN: {len(clusters)} clusters across "
      f"{clusters['scientific_name'].nunique()} species")

clusters_wgs = clusters.to_crs(epsg=4326)
gdf_wgs = gpd.GeoDataFrame(
    gdf, geometry=gpd.points_from_xy(gdf["x"], gdf["y"]), crs=gdf.crs
).to_crs(epsg=4326)

m = folium.Map(location=[39.9526, -75.1652], zoom_start=12, tiles="cartodbpositron")

for sp in clusters_wgs["scientific_name"].unique():
    sp_clusters = clusters_wgs[clusters_wgs["scientific_name"] == sp] \
        .sort_values("tree_count", ascending=False)
    sp_trees = gdf_wgs[gdf_wgs["scientific_name"] == sp]
    in_set = included_idx.get(sp, set())

    fg = folium.FeatureGroup(
        name=f"{sp} — {len(sp_clusters)} clusters / {len(sp_trees)} trees",
        show=False,
    )

    for row in sp_clusters.itertuples():
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x: {
                "fillColor": POLY_COLOR, "color": POLY_COLOR,
                "weight": 1.5, "fillOpacity": 0.3,
            },
            tooltip=(f"<b>{sp}</b><br>{row.tree_count} trees<br>"
                     f"{row.area_sqft/43560:.2f} acres<br>"
                     f"{row.density*43560:.1f} trees/acre"),
        ).add_to(fg)

    for idx, tree in sp_trees.iterrows():
        in_cluster = idx in in_set
        folium.CircleMarker(
            location=[tree.geometry.y, tree.geometry.x],
            radius=2,
            color=DOT_IN if in_cluster else DOT_OUT,
            fill=True,
            fill_color=DOT_IN if in_cluster else DOT_OUT,
            fill_opacity=0.85 if in_cluster else 0.5,
            weight=0,
        ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save("philly_tree_clusters_hdbscan.html")

/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` t

HDBSCAN: 428 clusters across 44 species
